In [10]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu langchain-huggingface

In [11]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
import faiss
from tqdm.auto import tqdm

In [12]:
# Upload Preprocessed Dataset
from google.colab import files
uploaded = files.upload()

# Loading the preprocessed dataset
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print(f"Loaded dataset with shape: {df.shape}")
display(df.sample(3))

Saving embedded_dataset.csv to embedded_dataset.csv
Loaded dataset with shape: (1003, 8)


,Question,Response,Domain,question_length,answer_length,Category,is_urgent,keyword_tags
191,Not feeling connected to my babywhats wrong wi...,Nothing is wrong with youyou may be experienci...,PPD,9,64,Relationship & Social Support,False,"feeling, connected, babywhats, wrong"
375,How can I improve my sleep after having a baby?,"Try to sleep when your baby sleeps, create a r...",PPD,10,19,Parenting & Baby Care Stress,False,"improve, sleep, baby"
726,"He isn't violent, but he has anger issues and ...",Sometimes relationships just do not work. Don...,Counsel_Chat,47,118,Follow-up & Continuous Support,False,"violent, anger, issues, deep, insecurities"


In [13]:
# Preparing Text for Embedding

# Ensuring all text inputs are strings and replacing NaNs
QnA = [(str(q) if pd.notna(q) else "") for q in (df['Question'] + " " + df['Response'])]

print("Text prepared for embedding.")


Text prepared for embedding.


In [14]:
# Loading Embedding Models

# MiniLM model for general-purpose semantic embeddings
model_general_name = "sentence-transformers/all-MiniLM-L6-v2"
model_general = HuggingFaceEmbeddings(
    model_name=model_general_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# PubMedBERT model for clinical domain-specific embeddings
model_medical_name = "neuml/pubmedbert-base-embeddings"
model_medical = HuggingFaceEmbeddings(
    model_name=model_medical_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("Embedding models loaded successfully.")


Embedding models loaded successfully.


In [16]:
# Generating and Storing Dual Embeddings

# Generating general-purpose embeddings
print("Generating general-purpose MiniLM embeddings...")
df['embedding_minilm'] = model_general.embed_documents(QnA)

# Generating domain-specific PubMedBERT embeddings
print("Generating medical-domain PubMedBERT embeddings...")
df['embedding_pubmed'] = model_medical.embed_documents(QnA)

print("Dual embeddings generated successfully.")


Generating general-purpose MiniLM embeddings...
Generating medical-domain PubMedBERT embeddings...
Dual embeddings generated successfully.


In [17]:
# Building and Saving FAISS Index for MiniLM (Cosine Similarity)

# Converting MiniLM embeddings into a NumPy array
embedding_matrix_minilm = np.array(df['embedding_minilm'].tolist()).astype('float32')

# Normalizing embeddings to unit length
faiss.normalize_L2(embedding_matrix_minilm)

# Creating FAISS index for cosine similarity
index_minilm = faiss.IndexFlatIP(embedding_matrix_minilm.shape[1])
index_minilm.add(embedding_matrix_minilm)

# Saving FAISS index
faiss.write_index(index_minilm, "faiss_minilm_cosine.index")

print("MiniLM FAISS index (cosine similarity) built and saved.")


MiniLM FAISS index (cosine similarity) built and saved.


In [18]:
# Building and Saving FAISS Index for PubMedBERT (Cosine Similarity)

# Converting PubMedBERT embeddings into a NumPy array
embedding_matrix_pubmed = np.array(df['embedding_pubmed'].tolist()).astype('float32')

# Normalizing embeddings to unit length
faiss.normalize_L2(embedding_matrix_pubmed)

# Creating FAISS index for cosine similarity
index_pubmed = faiss.IndexFlatIP(embedding_matrix_pubmed.shape[1])
index_pubmed.add(embedding_matrix_pubmed)

# Saving FAISS index
faiss.write_index(index_pubmed, "faiss_pubmed_cosine.index")

print("PubMedBERT FAISS index (cosine similarity) built and saved.")


PubMedBERT FAISS index (cosine similarity) built and saved.


In [19]:
# Saving the DataFrame

# Saving full DataFrame including both embeddings to a .pkl file
output_path = "dual_embeddings.pkl"
df.to_pickle(output_path)

print(f"Enriched dataset saved as {output_path}.")


Enriched dataset saved as dual_embeddings.pkl.
